# Landmark Detection Project
## Landmark Image Classification with a Vision Transformer Built from Scratch

This project identifies the landmark shown in an image using a **Vision Transformer (ViT)** trained entirely from scratch. It uses **no CNN, no convolutional layer, no pretrained backbone, and no transfer learning.**

We build a compact educational dataset containing five landmark classes:

1. Eiffel Tower
2. Taj Mahal
3. Statue of Liberty
4. Colosseum
5. Sydney Opera House

Images are downloaded from **exact landmark categories** on Wikimedia Commons (via its public API) instead of noisy keyword search results. This keeps the dataset small, clean, and easy to reason about while we build every piece of the Transformer ourselves.

> **Using your own dataset instead:** If you have your own dataset link/folder, you can skip Section 2 and instead point the `image_paths` / `labels` lists built at the end of Section 2 to your own files — everything after that (preprocessing, model, training, evaluation) stays the same.


## Why a Vision Transformer?

A digital image can be divided into small, non-overlapping **patches**. Each patch is flattened and projected into a learned token representation. **Positional embeddings** preserve the location of every patch, and **multi-head self-attention** learns relationships among all image regions — including ones far apart, which a small CNN kernel cannot easily do.

This notebook builds the following components ourselves, with plain TensorFlow/Keras layers:

- Image patch extraction
- Dense patch projection (patch embedding)
- Learnable positional embeddings
- Multi-head self-attention
- Feed-forward Transformer blocks
- Residual connections and layer normalization
- Global sequence pooling and a classification head

No CNN or pretrained visual feature extractor is used anywhere in the model.


## Section 1 — Imports and Reproducibility

Before doing anything else, we:

- Import the libraries we'll need (TensorFlow/Keras for the model, `requests` for the Wikimedia API, `PIL`/`numpy` for image handling, `sklearn` for train/test splitting and evaluation metrics).
- Fix random seeds so results are reproducible across runs.
- Check whether a GPU is available, since training a Transformer from scratch is much faster with one (Colab: `Runtime > Change runtime type > GPU`).


In [ ]:
import os
import io
import csv
import time
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))


## Section 2 — Dataset Download and Verification

We create an educational landmark dataset through the **Wikimedia Commons API**. We read files from *exact* landmark categories rather than broad search results, which reduces incorrect and weakly related training images.

**Dataset source:**
- Wikimedia Commons: https://commons.wikimedia.org/
- Wikimedia Commons API: https://commons.wikimedia.org/w/api.php
- API documentation: https://www.mediawiki.org/wiki/API:Main_page

For each class, we query the category's member files, filter to real image files (jpg/png), and download a fixed number of images. We save a `metadata.csv` with each downloaded image's class, Wikimedia title, local path, information page, and original file URL. **Individual Wikimedia files can have different licenses, so their information pages should be reviewed before any redistribution.**

> If you're plugging in your own dataset, replace this whole section with code that populates `DATA_DIR` with one subfolder per class, and skip straight to Section 3.


### Quick diagnostic (run this first if downloads return 0 images)

Before re-running the full download loop, run the cell below once. It tests three things separately and prints the exact status code / response so we can see *why* things are failing instead of guessing:

1. Basic internet access from this Colab runtime
2. A single Wikimedia Commons API call
3. A single Openverse API call

**How to read the output:**
- If step 1 fails -> this runtime has no/blocked outbound internet access (rare, but happens on some restricted Colab sessions) — try `Runtime > Disconnect and delete runtime`, reconnect, and re-run from the top.
- If step 1 works but steps 2 and 3 both show non-200 status codes or empty bodies -> this IP is being blocked by both APIs; wait a few minutes and retry, or use your own dataset instead (see the note at the top of Section 2).
- If step 2 or 3 shows status 200 with real JSON content -> the issue is in how we're calling that API (e.g. wrong parameter), not connectivity — share that output and we'll fix the specific call.


In [ ]:
print("=== 1. Basic internet connectivity ===")
try:
    r = requests.get("https://www.google.com", timeout=10)
    print("google.com status:", r.status_code)
except Exception as e:
    print("Basic internet test FAILED:", repr(e))

print("\n=== 2. Wikimedia Commons API ===")
try:
    r = requests.get(
        "https://commons.wikimedia.org/w/api.php",
        params={
            "action": "query", "list": "categorymembers",
            "cmtitle": "Category:Eiffel Tower", "cmtype": "file",
            "cmlimit": "5", "format": "json",
        },
        headers={"User-Agent": "LandmarkViT-Educational-Notebook/1.0 (contact: student@example.com)"},
        timeout=15,
    )
    print("status:", r.status_code, "| content-type:", r.headers.get("Content-Type"))
    print("body snippet:", r.text[:300])
except Exception as e:
    print("Commons API test FAILED:", repr(e))

print("\n=== 3. Openverse API ===")
try:
    r = requests.get(
        "https://api.openverse.org/v1/images/",
        params={"q": "Eiffel Tower", "page_size": 5},
        timeout=15,
    )
    print("status:", r.status_code, "| content-type:", r.headers.get("Content-Type"))
    print("body snippet:", r.text[:300])
except Exception as e:
    print("Openverse API test FAILED:", repr(e))


In [ ]:
DATA_DIR = "landmark_dataset"
IMAGES_PER_CLASS = 120  # keep this modest so the notebook runs quickly end-to-end

CATEGORIES = {
    "eiffel_tower": "Category:Eiffel Tower",
    "taj_mahal": "Category:Taj Mahal",
    "statue_of_liberty": "Category:Statue of Liberty",
    "colosseum": "Category:Colosseum",
    "sydney_opera_house": "Category:Sydney Opera House",
}

API_URL = "https://commons.wikimedia.org/w/api.php"
HEADERS = {"User-Agent": "LandmarkViT-Educational-Notebook/1.0 (contact: student@example.com)"}

os.makedirs(DATA_DIR, exist_ok=True)
metadata_rows = []

# A shared session + basic retry logic makes the notebook resilient to
# temporary rate-limiting or transient network errors from the API.
session = requests.Session()
session.headers.update(HEADERS)

def api_get_json(params, max_retries=4, base_delay=2.0):
    """GET from the Commons API and safely parse JSON, retrying on failures
    (rate limiting, empty/HTML responses, connection errors) instead of crashing."""
    for attempt in range(max_retries):
        try:
            resp = session.get(API_URL, params=params, timeout=30)
        except requests.exceptions.RequestException as e:
            print(f"  [retry {attempt+1}/{max_retries}] network error: {e}")
            time.sleep(base_delay * (attempt + 1))
            continue

        if resp.status_code == 200 and "application/json" in resp.headers.get("Content-Type", ""):
            try:
                return resp.json()
            except ValueError:
                pass  # fall through to retry

        print(f"  [retry {attempt+1}/{max_retries}] bad response "
              f"(status={resp.status_code}, ctype={resp.headers.get('Content-Type')})")
        time.sleep(base_delay * (attempt + 1))

    return None  # give up after all retries; caller should skip this item

def get_category_files(category, limit):
    """Return up to `limit` image-file titles that are direct members of a Commons category."""
    files = []
    cmcontinue = None
    while len(files) < limit:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": category,
            "cmtype": "file",
            "cmlimit": "100",
            "format": "json",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue
        resp = api_get_json(params)
        if resp is None:
            break
        members = resp.get("query", {}).get("categorymembers", [])
        files.extend(m["title"] for m in members)
        cmcontinue = resp.get("continue", {}).get("cmcontinue")
        time.sleep(0.3)
        if not cmcontinue:
            break
    return files[:limit]

def get_file_url(title):
    """Resolve a Commons file title to its direct image URL and description page."""
    params = {
        "action": "query",
        "titles": title,
        "prop": "imageinfo",
        "iiprop": "url",
        "format": "json",
    }
    resp = api_get_json(params)
    if resp is None:
        return None, None
    pages = resp.get("query", {}).get("pages", {})
    for page in pages.values():
        info = page.get("imageinfo")
        if info:
            return info[0]["url"], info[0].get("descriptionurl", "")
    return None, None

def download_class_commons(class_name, category, limit, verbose=True):
    """Primary path: pull directly from a Wikimedia Commons category."""
    class_dir = os.path.join(DATA_DIR, class_name)
    os.makedirs(class_dir, exist_ok=True)
    titles = get_category_files(category, limit * 3)  # over-fetch; some titles won't be usable images
    if verbose:
        print(f"  [commons] found {len(titles)} candidate file titles for {category}")

    saved = 0
    no_url = 0
    dl_failed = 0
    for title in titles:
        if saved >= limit:
            break
        if not title.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        url, desc_url = get_file_url(title)
        if not url:
            no_url += 1
            continue
        try:
            img_resp = session.get(url, timeout=30)
            img_resp.raise_for_status()
            img = Image.open(io.BytesIO(img_resp.content)).convert("RGB")
        except Exception as e:
            dl_failed += 1
            if verbose and dl_failed <= 3:  # don't flood the output
                print(f"    [commons] download/open failed for '{title}': {repr(e)}")
            continue
        fname = f"{class_name}_{saved:04d}.jpg"
        fpath = os.path.join(class_dir, fname)
        img.save(fpath, "JPEG", quality=90)
        metadata_rows.append({
            "class": class_name,
            "title": title,
            "local_path": fpath,
            "description_url": desc_url,
            "source_url": url,
        })
        saved += 1
        time.sleep(0.3)  # be polite to the API and avoid rate-limiting

    if verbose:
        print(f"  [commons] saved={saved}, no_url={no_url}, download_failed={dl_failed}")
    return saved

OPENVERSE_API = "https://api.openverse.org/v1/images/"

def download_class_openverse(class_name, query, limit, verbose=True):
    """Fallback path: Openverse indexes CC-licensed images (including Wikimedia
    Commons) and is much more tolerant of cloud/datacenter IPs than Wikimedia's
    own API/media servers sometimes are."""
    class_dir = os.path.join(DATA_DIR, class_name)
    os.makedirs(class_dir, exist_ok=True)
    saved = 0
    page = 1
    total_seen = 0
    dl_failed = 0
    while saved < limit and page <= 5:
        params = {
            "q": query,
            "page_size": 20,
            "page": page,
        }
        try:
            resp = session.get(OPENVERSE_API, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            if verbose:
                print(f"  [openverse] request failed (page {page}): {repr(e)}")
            break

        results = data.get("results", [])
        if verbose and page == 1:
            print(f"  [openverse] result_count={data.get('result_count')} for query '{query}'")
        if not results:
            break

        for item in results:
            if saved >= limit:
                break
            total_seen += 1
            url = item.get("url")
            if not url:
                continue
            try:
                img_resp = session.get(url, timeout=30)
                img_resp.raise_for_status()
                img = Image.open(io.BytesIO(img_resp.content)).convert("RGB")
            except Exception as e:
                dl_failed += 1
                if verbose and dl_failed <= 3:
                    print(f"    [openverse] download/open failed: {repr(e)}")
                continue
            fname = f"{class_name}_{saved:04d}.jpg"
            fpath = os.path.join(class_dir, fname)
            img.save(fpath, "JPEG", quality=90)
            metadata_rows.append({
                "class": class_name,
                "title": item.get("title", ""),
                "local_path": fpath,
                "description_url": item.get("foreign_landing_url", ""),
                "source_url": url,
            })
            saved += 1
            time.sleep(0.2)
        page += 1

    if verbose:
        print(f"  [openverse] saved={saved}, candidates_tried={total_seen}, download_failed={dl_failed}")
    return saved

def download_class(class_name, category, limit):
    print(f"--- {class_name} ---")
    saved = download_class_commons(class_name, category, limit)
    if saved == 0:
        print(f"  Wikimedia Commons returned 0 usable images -> falling back to Openverse")
        query = category.replace("Category:", "")
        saved = download_class_openverse(class_name, query, limit)
    print(f"{class_name}: TOTAL saved {saved} images\n")
    return saved

for class_name, category in CATEGORIES.items():
    download_class(class_name, category, IMAGES_PER_CLASS)

metadata_df = pd.DataFrame(metadata_rows)

if len(metadata_df) == 0:
    raise RuntimeError(
        "No images were downloaded from either Wikimedia Commons or Openverse. "
        "Scroll up and check the [commons]/[openverse] debug lines above for the "
        "specific failure reason (no_url, download_failed exceptions, or 0 search "
        "results), or plug in your own dataset folder here instead (see the note "
        "at the top of Section 2)."
    )

metadata_df.to_csv("metadata.csv", index=False)
print("\nTotal images downloaded:", len(metadata_df))
metadata_df.head()


In [ ]:
# Quick verification: make sure every class folder actually has images, and preview a few
for class_name in CATEGORIES:
    class_dir = os.path.join(DATA_DIR, class_name)
    n = len(os.listdir(class_dir)) if os.path.isdir(class_dir) else 0
    print(f"{class_name}: {n} images")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, class_name in zip(axes, CATEGORIES):
    class_dir = os.path.join(DATA_DIR, class_name)
    files = os.listdir(class_dir)
    if files:
        img = Image.open(os.path.join(class_dir, files[0]))
        ax.imshow(img)
    ax.set_title(class_name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Section 3 — Preprocessing and Data Pipeline

Photos vary wildly in size and aspect ratio, so we:

1. **Resize with aspect-ratio preservation** by padding to a square before resizing, instead of squashing images — this avoids distorting the landmark's shape.
2. Build a train / validation / test split (stratified by class).
3. Wrap everything in a `tf.data.Dataset` pipeline with batching, shuffling, and prefetching.
4. Apply light **data augmentation** (flip, rotation, zoom, contrast) only to the training set, since our dataset is small and a Transformer trained from scratch has no built-in image bias — it needs the extra examples.


In [ ]:
IMG_SIZE = 96      # patch grid resolution; keep modest since we train from scratch on a small dataset
PATCH_SIZE = 8
NUM_CLASSES = len(CATEGORIES)
CLASS_NAMES = list(CATEGORIES.keys())

def load_and_pad_resize(path, size=IMG_SIZE):
    img = Image.open(path).convert("RGB")
    img = ImageOps.pad(img, (size, size), color=(0, 0, 0))  # preserves aspect ratio, pads with black
    return np.array(img, dtype=np.uint8)

image_paths = metadata_df["local_path"].tolist()
labels = [CLASS_NAMES.index(c) for c in metadata_df["class"].tolist()]

X = np.stack([load_and_pad_resize(p) for p in image_paths])
y = np.array(labels)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

def make_dataset(X, y, training, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(len(X), seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val, training=False)
test_ds = make_dataset(X_test, y_test, training=False)


## Section 4 — Building the Vision Transformer From Scratch

This is the core of the project. We implement each ViT component as its own Keras layer, then assemble them:

- **Patch extraction**: split each image into fixed-size, non-overlapping patches using `tf.image.extract_patches`.
- **Patch + position embedding**: flatten each patch and linearly project it into a `d_model`-dimensional token; add a learnable positional embedding per patch index, plus a learnable `[CLS]` token that will represent the whole image.
- **Transformer encoder block**: layer norm → multi-head self-attention → residual add → layer norm → MLP → residual add.
- **Classification head**: take the `[CLS]` token's final representation and pass it through a small dense head.


In [ ]:
class PatchExtractor(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches  # (batch, num_patches, patch_size*patch_size*channels)


class PatchEmbedding(layers.Layer):
    def __init__(self, num_patches, d_model, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection = layers.Dense(d_model)
        self.cls_token = self.add_weight(
            name="cls_token", shape=(1, 1, d_model), initializer="random_normal", trainable=True
        )
        self.position_embedding = layers.Embedding(input_dim=num_patches + 1, output_dim=d_model)

    def call(self, patches):
        batch_size = tf.shape(patches)[0]
        projected = self.projection(patches)  # (batch, num_patches, d_model)
        cls_tokens = tf.broadcast_to(self.cls_token, [batch_size, 1, tf.shape(projected)[-1]])
        tokens = tf.concat([cls_tokens, projected], axis=1)  # prepend CLS token
        positions = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        tokens = tokens + self.position_embedding(positions)
        return tokens


class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, mlp_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = keras.Sequential([
            layers.Dense(mlp_dim, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(d_model),
            layers.Dropout(dropout),
        ])

    def call(self, x, training=False):
        x_norm = self.norm1(x)
        attn_out = self.attn(x_norm, x_norm, training=training)
        x = x + attn_out  # residual connection
        x_norm2 = self.norm2(x)
        mlp_out = self.mlp(x_norm2, training=training)
        x = x + mlp_out  # residual connection
        return x


def build_vit(image_size, patch_size, num_classes, d_model=128, depth=6,
              num_heads=4, mlp_dim=256, dropout=0.1):
    num_patches = (image_size // patch_size) ** 2

    inputs = layers.Input(shape=(image_size, image_size, 3))
    x = PatchExtractor(patch_size)(inputs)
    x = PatchEmbedding(num_patches, d_model)(x)
    x = layers.Dropout(dropout)(x)

    for _ in range(depth):
        x = TransformerBlock(d_model, num_heads, mlp_dim, dropout)(x)

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    cls_output = x[:, 0, :]  # the [CLS] token summarizes the whole image
    x = layers.Dense(mlp_dim, activation="gelu")(cls_output)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="vision_transformer_from_scratch")

vit_model = build_vit(IMG_SIZE, PATCH_SIZE, NUM_CLASSES)
vit_model.summary()


## Section 5 — Training

We compile the model with Adam, sparse categorical cross-entropy (since our labels are integers, not one-hot), and accuracy as the tracked metric. We use `EarlyStopping` and `ReduceLROnPlateau` since a from-scratch Transformer on a small dataset can overfit quickly.


In [ ]:
vit_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].legend()
plt.show()


## Section 6 — Evaluation

We check performance on the held-out **test set** (images the model never saw during training or validation), and inspect a confusion matrix to see which landmarks get confused with each other.


In [ ]:
test_loss, test_acc = vit_model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

y_pred_probs = vit_model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
plt.colorbar()
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## Section 7 — Test-Time Augmentation (TTA)

To squeeze out a bit more robustness at inference time, we predict on several augmented versions of each test image (e.g. flipped, slightly rotated) and average the resulting probabilities.


In [ ]:
def predict_with_tta(model, image_array, n_augments=5):
    """image_array: single image, shape (IMG_SIZE, IMG_SIZE, 3), values in [0, 255]"""
    img = tf.cast(image_array, tf.float32) / 255.0
    img = tf.expand_dims(img, 0)

    preds = [model.predict(img, verbose=0)]
    for _ in range(n_augments - 1):
        aug_img = data_augmentation(img, training=True)
        preds.append(model.predict(aug_img, verbose=0))

    avg_pred = np.mean(preds, axis=0)
    return avg_pred[0]

# Example: TTA on a handful of test images
for i in range(3):
    probs = predict_with_tta(vit_model, X_test[i])
    pred_class = CLASS_NAMES[np.argmax(probs)]
    true_class = CLASS_NAMES[y_test[i]]
    print(f"True: {true_class:20s}  Predicted (TTA): {pred_class:20s}  Confidence: {np.max(probs):.2f}")


## Section 8 — Live Prediction and Interactive Dashboard

Finally, we wrap the model in a small interactive widget so you can upload your own photo of one of the five landmarks and see the model's prediction, confidence scores for every class, and a bar chart — all inside the notebook.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

def predict_uploaded_image(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img_resized = ImageOps.pad(img, (IMG_SIZE, IMG_SIZE), color=(0, 0, 0))
    arr = np.array(img_resized, dtype=np.uint8)
    probs = predict_with_tta(vit_model, arr, n_augments=5)

    pred_idx = np.argmax(probs)
    print(f"Prediction: {CLASS_NAMES[pred_idx]}  (confidence: {probs[pred_idx]:.2%})\n")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_resized)
    axes[0].axis("off")
    axes[0].set_title("Input image")

    axes[1].barh(CLASS_NAMES, probs)
    axes[1].set_xlim(0, 1)
    axes[1].set_title("Class probabilities")
    plt.tight_layout()
    plt.show()

print("Upload a photo of one of: Eiffel Tower, Taj Mahal, Statue of Liberty, Colosseum, Sydney Opera House")
uploaded = files.upload()
for fname, content in uploaded.items():
    print(f"\n--- {fname} ---")
    predict_uploaded_image(content)
